# C10-competition-craft — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: the concept summary, the craft checklists
you should run before any submission, a self-quiz spanning every
taught concept, and pointers to what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.

In [ ]:
import numpy as np
import pandas as pd

## Concept summary

| Concept | One-line summary | Key fact to retain | Session |
|---|---|---|---|
| `prediction-function-contract` | `predict_labels(X_test) -> pd.Series`: any-length feature DataFrame in; Series out — same length, same index, original order, training-vocabulary values | the grader aligns by index; any clause violation scores **zero** regardless of model quality | 1 |
| `hidden-test-protocol` | the held-back split is regenerated deterministically for grading; your notebook never reads it | "hidden" is a protocol, not encryption — every reported number must be computable without the held-back rows | 1 |
| `metric-driven-iteration` | macro-F1 (mean of per-class one-vs-rest F1s) judged on a frozen seeded stratified carve; baseline → error analysis → one change → re-validate, logged and capped | macro weighs classes equally, so accuracy > macro-F1 whenever the minority lags; selection on the carve inflates the estimate | 2 |
| `notebook-discipline` | pin seeds, cell order = execution order, no dead cells, deterministic re-run | the audit: wrap the whole flow in a function, run twice, demand **exact** equality | 3 |
| `writeup-quality` | one markdown cell: approach / intuition / alternatives, every claim backed by a printed number | the six-point rubric — recipe+protocol, data-grounded and task-grounded reasons, alternative-with-outcome + limitation | 3 |

## The pre-submission checklists

**The contract self-check (run on a mid-table probe):**

```python
probe = X.iloc[300:340]                  # NOT the top of the table
out = predict_labels(probe)
isinstance(out, pd.Series)               # R3
len(out) == len(probe)                   # R4 length
out.index.equals(probe.index)            # R4 index
set(out.unique()) <= set(np.unique(y))   # R5 vocabulary
```

**Macro-F1 from the confusion matrix (rows=actual, cols=predicted,
sorted label order):**

```python
diag = np.diag(C).astype(float)
prec = diag / C.sum(axis=0)              # column sums: predicted-as-k
rec  = diag / C.sum(axis=1)              # row sums: actually-k
f1   = 2 * prec * rec / (prec + rec)
macro = f1.mean()
```

**The frozen carve (this unit's pinned protocol):**

```python
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)
```

**The determinism audit:**

```python
def run_submission():
    ...                                  # carve, fit, evaluate, predict
    return val_f1, probe_preds

a, b = run_submission(), run_submission()
assert a[0] == b[0] and (a[1] == b[1]).all()    # EXACT, not isclose
```

**The discipline rules:** D1 pin every seed · D2 cell order =
execution order · D3 no dead cells · D4 deterministic re-run
(Restart & Run All, twice, compare).

**The writeup rubric (score your own before submitting):**
W-A recipe with values / protocol with score · W-B model reason from
the data / metric reason from the task · W-C alternative with
outcome / honest limitation.

## Self-quiz

Work every item on paper (and NumPy where asked) *before* opening the
answers at the end.

1. Recite the five contract clauses (R1–R5) with one violation each.
2. `predict_labels` returns
   `pd.Series(pipe.predict(X_test)).reset_index(drop=True)`.
   On which probes does the self-check *pass*, and why is the
   function still a zero waiting to happen?
3. Why does the final submitted model get refit on **all** labeled
   rows, and what is the reported `val_f1` then an estimate of?
4. State the hidden-test protocol in two sentences: one about you,
   one about the graders.
5. A teammate argues: "the held-back split is deterministic, so
   reading it once to sanity-check my score harms nobody."
   Give the two-part rebuttal.
6. Confusion matrix (rows/cols in order `struggles`, `thrives`):
   $\begin{pmatrix} 12 & 8 \\ 5 & 35 \end{pmatrix}$.
   Compute both per-class F1s, the macro-F1, and the accuracy, as
   exact fractions.
7. Why is the do-nothing (always-majority) rule's macro-F1 so much
   lower than its accuracy on a 2:1 task?
   Give both numbers for a 100-row validation set split 67/33.
8. What two properties must the validation carve have on an
   imbalanced task, and what C4 pitfall does *re-carving with new
   seeds until the number improves* commit?
9. State the overfitting-to-validation fact, and the three-word
   mechanism.
10. Your iteration log shows gains of +0.004, +0.006, +0.003 over
    ten tweaks.
    The seed-luck spread is ~0.05.
    What does Session 2 §6 say about the total?
11. Name the four discipline rules and, for each, the failure it
    prevents at grading time.
12. Why does the determinism audit use `==` instead of `isclose`,
    and in what setting would a stated tolerance (atol, rtol=0) be
    the right tool instead?
13. Score this writeup fragment against all six rubric points:
    "Scaled 7-NN pipeline on all 12 features; stratified seeded
    150-row carve; validation macro-F1 0.79.
    kNN suits the data because scaled sensor profiles cluster by
    outcome; macro-F1 suits the task because classes are 2:1 and
    the minority matters most.
    We tried k = 3 (0.78) and k = 11 (0.81) — wait, and submitted
    k = 7 anyway."
    (Read carefully.)
14. What is the *first* thing to suspect when a submission scores
    far below its `val_f1`, and what is the *second*?
15. In one sentence each: what do the mock tests add on top of this
    unit's mini-competition, and which unit artifacts should you
    bring to them as reflexes?

## What to redo, per weak spot

| Shaky on … | Redo |
|---|---|
| contract clauses + self-checks | Session 1 §3, §5; `p01`, `p05`, `p08` |
| hidden-test protocol | Session 1 §4, §6; `p02`, `p09`, `p16` |
| macro-F1 arithmetic | Session 2 §1–3; `p04`, `p07`, `p11` |
| validation honesty + iteration | Session 2 §4–6; `p03`, `p12`, `p13` |
| discipline + determinism | Session 3 §1–2; `p06`, `p10`, `p14` |
| the writeup rubric | Session 3 §3–4; `p15`, `p18` |
| the whole arc, timed | Session 3 §5; `p17` — then `mocktests/` |

## Quiz answers

<details><summary><b>Answers 1–15</b> (open only after committing to yours)</summary>

1. R1 exact name `predict_labels` exists (typo, conditional def);
   R2 accepts any length (hardcoded row count); R3 returns
   `pd.Series` (list/ndarray/DataFrame); R4 length **and** index
   equal the input's, order preserved (dropping rows, `reset_index`,
   sorting); R5 values from the training vocabulary (0/1 codes for
   string labels).
2. It passes on probes whose index already is `0..n-1` (e.g. the top
   of the table) — `reset_index` then reproduces the same index by
   coincidence.
   Any mid-table probe (index 300..339) exposes the R4 index
   violation; the grader's rows won't be indexed 0..n-1 either.
3. Every labeled row left out of the final fit is signal the
   grader's rows never benefit from; `val_f1` is an estimate of the
   *recipe's* held-back performance, measured on a smaller fit of
   the same recipe — typically slightly conservative, and stated as
   such in the writeup.
4. You: never read, print, or use the held-back rows; every number
   you report is computable without them.
   Graders: regenerate the split deterministically with the seeded
   script's grading flag and score `predict_labels` on it, once.
5. (i) The habit is the exam skill — on the real paper the rows are
   genuinely unreachable, so practicing the peek trains the wrong
   reflex; (ii) any number influenced by held-back rows stops
   estimating generalization — it is leakage at answer-key strength,
   and the automated sweep zeroes it regardless of intent.
6. `struggles`: prec $= 12/17$, rec $= 12/20 = 3/5$,
   $F_1 = \frac{2 \cdot \frac{12}{17} \cdot \frac35}{\frac{12}{17} + \frac35}
   = \frac{72/85}{111/85} = \frac{72}{111} = \frac{24}{37}$.
   `thrives`: prec $= 35/43$, rec $= 35/40 = 7/8$,
   $F_1 = \frac{2 \cdot \frac{35}{43} \cdot \frac78}{\frac{35}{43} + \frac78}
   = \frac{490/344}{581/344} = \frac{490}{581} = \frac{70}{83}$.
   Macro-F1 $= \frac12(\frac{24}{37} + \frac{70}{83}) =
   \frac12 \cdot \frac{24 \cdot 83 + 70 \cdot 37}{3071} =
   \frac{4582}{6142} = \frac{2291}{3071}$ ($\approx 0.746$).
   Accuracy $= 47/60$ ($\approx 0.783$).
7. The rule never finds the minority: minority F1 $= 0$, and macro
   averaging gives that zero half the weight; accuracy only charges
   it the minority's 33 rows.
   Numbers: accuracy $= 0.67$; majority F1
   $= \frac{2 \cdot 0.67 \cdot 1}{1.67} \approx 0.802$, so macro-F1
   $\approx 0.401$.
8. Seeded (frozen) and stratified.
   Re-carving until the number improves is split shopping — C4's
   Pitfall 4 — and reports an upward-biased order statistic.
9. Every candidate comparison decided by the validation score makes
   the surviving score an optimistic estimate of held-back
   performance, growing with the number of candidates.
   Mechanism: max of noise.
10. The +0.013 total is far inside the ~0.05 seed-luck spread and
    was accumulated over ten selections — it is weather, selected;
    expect the held-back score to give most of it back.
11. D1 pin seeds (results change between runs); D2 order =
    execution (fresh-kernel run crashes or silently recomputes);
    D3 no dead cells (one `NameError` fails the run-clean
    component); D4 deterministic re-run (claims in the writeup
    match what the grader sees).
12. Same machine, same environment, same seeds: a pure function —
    exact equality is owed, and a tolerance would mask real
    nondeterminism.
    Cross-machine/version anchor checks are where the course's
    stated-atol (rtol=0) contract belongs.
13. W-A1 ✓ (recipe with values and feature set), W-A2 ✓ (protocol +
    score), W-B1 ✓, W-B2 ✓, W-C1 ✓ (two alternatives with
    numbers)… and W-C2 ✗ (no limitation/next step) — but the
    *content* fails something the rubric can see: k = 11 scored
    0.81 > 0.79, so the log contradicts the submission ("submitted
    k = 7 anyway" with no stated reason).
    Score 5/6, and a grader will (rightly) distrust W-A2's claim of
    an honest protocol — internal consistency is part of quality.
14. First: a contract violation (run the four self-checks — most
    zeros are packaging); second: overfitting-to-validation (a long
    uncapped iteration loop — check the log's length and the size of
    accepted gains).
15. Mocks add the full exam texture — many problems, a time budget,
    and the applied task embedded among them; bring the contract
    self-check, the frozen-carve habit, the audit, and the rubric as
    automatic reflexes.

</details>